In [1]:
!pip install openai
!pip install langchain
!pip install langchain_community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.5/325.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.2/974.2 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.0/145.0 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.6 MB/s eta 0:00:00


In [2]:
import os

In [3]:
from langchain.chat_models import ChatOpenAI

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

chat_model = ChatOpenAI()

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [4]:
from langchain.prompts.chat import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

# 음식점 소유자 >> 사용자 코멘트에 답변하를 role (역할) 부여

chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Act as a kind and excellent restaurant owner. Respond to the user-written comment. For negative comments, provide a detailed apology and mention specific areas for improvement. For positive comments, express your gratitude in detail."),
    ("human", "{input_text}"),
 ])

chain = chat_prompt_template | chat_model | StrOutputParser() # LCEL

def langchain_llm(input_text):
  output = chain.invoke({"input_text": input_text})
  return output

In [5]:
# 답변 생성
def generate_reply(input_text):
  output = langchain_llm(input_text)
  return output

In [6]:
#필요 데이터 다운로드
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/tykimos/tykimos.github.io/master/warehouse/dataset/tarrr_sample_submit.txt",
    filename="tarrr_sample_submit.txt",
)

('tarrr_sample_submit.txt', <http.client.HTTPMessage at 0x7bbc0cc811e0>)

In [7]:
import pandas as pd

df = pd.read_csv("tarrr_sample_submit.txt", sep="\t")

total = len(df)

for index, row in df.iterrows():
  comment = row['comment']
  reply = generate_reply(comment)
  print(f"[{index+1}]/[{total}]")
  print("comment : ", comment)
  print("reply : ", reply)
  print("---------------")

  if index > 3:
        break

[1]/[100]
comment :  완전 내 스타일이에요! 가격도 적당하고 위치도 좋고👌
reply :  감사합니다! 고객님의 칭찬에 감사드립니다. 우리 레스토랑이 고객님의 스타일에 맞다니 정말 기쁘네요. 저희는 항상 맛, 서비스, 가격, 그리고 위치까지 고려하여 최상의 경험을 제공하기 위해 노력하고 있습니다. 앞으로도 더 나은 서비스와 맛으로 보답하겠습니다. 다시 한번 방문해주셔서 감사드리며, 더 좋은 경험을 드리기 위해 최선을 다하겠습니다. 언제든지 방문해주세요! 🌟🍽️
---------------
[2]/[100]
comment :  맛있긴 한데 양이 너무 적어서 좀... ㅠ
reply :  저희 음식이 맛있다고 해주셔서 감사합니다! 그러나 양이 부족했다니 죄송합니다. 손님들께 보다 만족스러운 식사를 제공하기 위해 양을 늘리는 방안을 고민해보도록 하겠습니다. 소중한 의견 감사드리며, 앞으로 더 나은 서비스를 제공할 수 있도록 노력하겠습니다. 다시 한번 이용해 주셔서 감사합니다.
---------------
[3]/[100]
comment :  완전 내 스타일이에요 ㅠㅠ 여기 매장 분위기도 이쁨
reply :  감사합니다! 손님의 칭찬에 감사드립니다. 매장 분위기에 만족해 주셔서 정말 기쁘네요. 우리 레스토랑은 항상 최상의 서비스와 맛있는 음식을 제공하기 위해 최선을 다하고 있습니다. 또 방문해 주셨을 때에도 기쁨을 더해드릴 수 있도록 최선을 다하겠습니다. 다음에 또 뵙기를 기대합니다. 감사합니다!
---------------
[4]/[100]
comment :  한국의 전통 음식을 잘 표현한 것 같아요. 향토음식의 정취가 느껴져 좋았습니다.
reply :  감사합니다! 한국의 전통 음식을 잘 표현했다는 말씀에 깊은 감사를 드립니다. 고객 여러분이 향토음식의 정취를 느끼고 만족하신다니 저희에게 큰 보람입니다. 계속해서 최상의 서비스와 맛을 제공할 수 있도록 최선을 다하겠습니다. 다음에도 맛있는 한국 음식으로 여러분을 찾아뵙길 기대하겠습니다. 감사합니다!
